# 2. BERT at its best configuration

**NLP Final Term Project, Group 02.**

Fine-tunes `bert-base-uncased` on both datasets, at the configuration that scored
highest on **validation** weighted F1 in the hyperparameter grid.

The grid was eight configurations per model per dataset: learning rate in
{2e-5, 3e-5}, batch size in {16, 32}, weight decay in {0.01, 0.1}; five epochs with
early stopping on validation F1, patience 2; maximum sequence length 128. The winner
was picked on validation and only then run against test, once. The test set selects
nothing.

| dataset | learning rate | batch size | weight decay |
|---|---|---|---|
| D1 DAIGT V2 | 3e-5 | 32 | 0.1 |
| D2 HC3 | 2e-5 | 16 | 0.1 |

**Requires** `01_preprocessing.ipynb` to have run. If the metrics and probabilities
for a configuration already exist on disk, the run is loaded rather than repeated,
so this notebook re-executes in seconds; delete the matching files in
`experiments/paper_scale/results/` and `probs/`, or pass `force=True`, to retrain.

## 2.1 Environment and paths

In [1]:
import os

os.environ.setdefault('HF_HOME', '/media/filwel/MLProject1/hf_cache')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import gc
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Raw corpora live outside the repository; the repository holds the derived splits.
PROJECT_DIR = Path('/media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Project ')

FINAL_DIR = Path('/media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final')
if not FINAL_DIR.exists():
    p = Path.cwd().resolve()
    while p.name != 'Final' and p != p.parent:
        p = p.parent
    FINAL_DIR = p

PS_DIR = FINAL_DIR / 'experiments' / 'paper_scale'
WORK_DIR = PS_DIR / 'work'
RESULTS_DIR = PS_DIR / 'results'
PROBS_DIR = PS_DIR / 'probs'
MODELS_DIR = PS_DIR / 'models'
CKPT_DIR = Path('/media/filwel/MLProject1/nlp_paper_ckpt')

MAX_LEN = 128
EPOCHS = 5
WARMUP_RATIO = 0.1
PATIENCE = 2
SPLIT_SEED = 42
TRAIN_SEED = 42

MODELS = {'BERT': 'bert-base-uncased', 'DeBERTa': 'microsoft/deberta-v3-base'}
DATASET_NAMES = {'D1': 'DAIGT V2', 'D2': 'HC3'}

## 2.2 The fixed split from notebook 01

In [2]:
def load_fixed_split(tag):
    """Load the one fixed split written by notebook 01.

    The split is built once, with seed 42, and reused by every training run.
    Only model initialisation and batch order vary with the training seed, never
    which rows sit in which partition; re-splitting per seed would silently move
    the evaluation set between runs and invalidate any across-seed comparison.
    """
    data_p = WORK_DIR / f'data_{tag}.parquet'
    split_p = WORK_DIR / f'split_{tag}.npz'
    if not (data_p.exists() and split_p.exists()):
        raise FileNotFoundError(
            f'missing {data_p.name} / {split_p.name}. Run 01_preprocessing.ipynb first.')
    df = pd.read_parquet(data_p)
    sp = np.load(split_p)
    return df, {'train': sp['train'], 'val': sp['val'], 'test': sp['test']}


DATA, SPLITS = {}, {}
for tag in ('D1', 'D2'):
    DATA[tag], SPLITS[tag] = load_fixed_split(tag)
    n = {k: len(v) for k, v in SPLITS[tag].items()}
    print(f'{tag} {DATASET_NAMES[tag]:9s} train={n["train"]:6d}  val={n["val"]:5d}  '
          f'test={n["test"]:6d}  total={sum(n.values()):6d}')

D1 DAIGT V2  train= 25196  val= 2800  test=  6998  total= 34994
D2 HC3       train= 38785  val= 4289  test= 10732  total= 53806


## 2.3 Tokenisation

In [3]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

_TOKCACHE, _DATACACHE = {}, {}


def normalise(t):
    return re.sub(r'\s+', ' ', str(t)).strip()


def get_tokenizer(model_key):
    if model_key not in _TOKCACHE:
        _TOKCACHE[model_key] = AutoTokenizer.from_pretrained(MODELS[model_key])
    return _TOKCACHE[model_key]


def get_tokenized(tag, model_key):
    """Tokenise the three partitions of one dataset for one model, cached in memory."""
    key = (tag, model_key)
    if key in _DATACACHE:
        return _DATACACHE[key]
    df, splits = DATA[tag], SPLITS[tag]
    tok = get_tokenizer(model_key)
    parts = {}
    for split, idx in splits.items():
        sub = df.loc[idx]
        ds = Dataset.from_dict({'text': [normalise(t) for t in sub['text']],
                                'labels': [int(v) for v in sub['label']]})
        parts[split] = ds.map(
            lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN),
            batched=True, remove_columns=['text'])
    _DATACACHE[key] = (parts, splits)
    gc.collect()
    return _DATACACHE[key]


print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| bf16', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

torch 2.11.0+cu128 | cuda True | bf16 True


## 2.4 Training harness

Weighted precision, recall and F1 are used throughout. The classes are balanced by
construction, so weighted and macro averaging agree closely, but weighted is what the
midterm reported and the two halves of the project stay comparable this way.

Early stopping monitors validation F1 with patience 2, and the best checkpoint by
that metric is what gets evaluated, never the last epoch.

In [4]:
import shutil
import time

from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)
from transformers import (AutoModelForSequenceClassification,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)

for d in (RESULTS_DIR, PROBS_DIR, MODELS_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)


def weighted_metrics(y, p):
    acc = accuracy_score(y, p)
    pre, rec, f1, _ = precision_recall_fscore_support(
        y, p, average='weighted', zero_division=0)
    return acc, pre, rec, f1


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc, pre, rec, f1 = weighted_metrics(labels, preds)
    return {'accuracy': acc, 'precision': pre, 'recall': rec, 'f1': f1}


def run_key(tag, model_key, cfg, seed=TRAIN_SEED):
    return (f'full_{tag}_{model_key}_lr{cfg["lr"]:g}_bs{cfg["bs"]}'
            f'_wd{cfg["wd"]:g}_s{seed}')


def train_one(tag, model_key, cfg, seed=TRAIN_SEED, save_model=True, force=False):
    """Fine-tune one configuration and write its metrics and probabilities.

    If both artefacts already exist and force is False the run is skipped and the
    stored record returned, so re-executing this notebook costs seconds instead of
    hours while still reporting the exact numbers the report quotes.
    """
    key = run_key(tag, model_key, cfg, seed)
    jpath, ppath = RESULTS_DIR / f'{key}.json', PROBS_DIR / f'{key}.npz'
    if not force and jpath.exists() and ppath.exists():
        rec = json.load(open(jpath))
        print(f'[cached] {key}  val_f1={rec["val"]["f1"]:.4f}  test_f1={rec["test"]["f1"]:.4f}')
        return rec

    run_dir = CKPT_DIR / key
    parts, splits = get_tokenized(tag, model_key)
    tok = get_tokenizer(model_key)

    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODELS[model_key], num_labels=2)
    model.config.id2label = {0: 'human', 1: 'ai'}
    model.config.label2id = {'human': 0, 'ai': 1}

    args = TrainingArguments(
        output_dir=str(run_dir),
        learning_rate=cfg['lr'],
        per_device_train_batch_size=cfg['bs'],
        per_device_eval_batch_size=64,
        weight_decay=cfg['wd'],
        num_train_epochs=EPOCHS,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type='linear',
        optim='adamw_torch',
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_f1',
        greater_is_better=True,
        logging_steps=200,
        seed=seed,
        data_seed=seed,
        dataloader_num_workers=0,
        report_to='none')

    trainer = Trainer(
        model=model, args=args, train_dataset=parts['train'],
        eval_dataset=parts['val'], data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])

    t0 = time.time()
    trainer.train()
    train_secs = time.time() - t0

    out = {'key': key, 'dataset': tag, 'dataset_name': DATASET_NAMES[tag],
           'model': model_key, 'checkpoint': MODELS[model_key],
           'lr': cfg['lr'], 'batch_size': cfg['bs'], 'weight_decay': cfg['wd'],
           'seed': seed, 'max_len': MAX_LEN,
           'n_train': len(splits['train']), 'n_val': len(splits['val']),
           'n_test': len(splits['test']), 'train_seconds': round(train_secs, 1),
           'epochs_run': int(trainer.state.epoch or 0)}

    probs = {}
    for split in ('val', 'test'):
        pred = trainer.predict(parts[split])
        raw = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
        p = torch.softmax(torch.tensor(raw, dtype=torch.float32), dim=-1).numpy()
        y = np.asarray(pred.label_ids)
        acc, pre, rec, f1 = weighted_metrics(y, p.argmax(1))
        out[split] = {'accuracy': round(acc, 4), 'precision': round(pre, 4),
                      'recall': round(rec, 4), 'f1': round(f1, 4)}
        out[f'{split}_confusion'] = confusion_matrix(y, p.argmax(1)).tolist()
        probs[f'{split}_probs'] = p
        probs[f'{split}_labels'] = y

    np.savez(ppath, **probs)
    json.dump(out, open(jpath, 'w'), indent=2)

    if save_model:
        mdir = MODELS_DIR / f'{tag}_{model_key}'
        trainer.save_model(str(mdir))
        tok.save_pretrained(str(mdir))
        json.dump(out, open(mdir / 'run_info.json', 'w'), indent=2)

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(run_dir, ignore_errors=True)
    print(f'[trained] {key}  val_f1={out["val"]["f1"]:.4f}  '
          f'test_f1={out["test"]["f1"]:.4f}  {train_secs / 60:.1f} min')
    return out

## 2.5 Best BERT configuration, per dataset

In [5]:
BERT_BEST = {
    'D1': {'lr': 3e-5, 'bs': 32, 'wd': 0.1},
    'D2': {'lr': 2e-5, 'bs': 16, 'wd': 0.1},
}

BERT_RESULT = {}
for tag in ('D1', 'D2'):
    cfg = BERT_BEST[tag]
    print(f'{tag} {DATASET_NAMES[tag]:9s} BERT  learning_rate={cfg["lr"]:g}  '
          f'batch_size={cfg["bs"]}  weight_decay={cfg["wd"]:g}')
    BERT_RESULT[tag] = train_one(tag, 'BERT', cfg, save_model=True)

D1 DAIGT V2  BERT  learning_rate=3e-05  batch_size=32  weight_decay=0.1
[cached] full_D1_BERT_lr3e-05_bs32_wd0.1_s42  val_f1=0.9957  test_f1=0.9916
D2 HC3       BERT  learning_rate=2e-05  batch_size=16  weight_decay=0.1
[cached] full_D2_BERT_lr2e-05_bs16_wd0.1_s42  val_f1=0.9939  test_f1=0.9916


## 2.6 Results

In [6]:
def result_table(records, title):
    rows = []
    for tag, rec in records.items():
        rows.append({'dataset': f'{tag} {DATASET_NAMES[tag]}',
                     'learning_rate': rec['lr'], 'batch_size': rec['batch_size'],
                     'weight_decay': rec['weight_decay'],
                     'epochs_run': rec.get('epochs_run'),
                     'val_f1': rec['val']['f1'],
                     'test_accuracy': rec['test']['accuracy'],
                     'test_precision': rec['test']['precision'],
                     'test_recall': rec['test']['recall'],
                     'test_f1': rec['test']['f1']})
    tab = pd.DataFrame(rows)
    print(title)
    print(tab.to_string(index=False))
    return tab


def show_confusion(records):
    for tag, rec in records.items():
        cm = np.array(rec['test_confusion'])
        print(f'{tag} {DATASET_NAMES[tag]} test confusion (rows true, cols predicted, '
              f'order human, ai)')
        print(pd.DataFrame(cm, index=['true_human', 'true_ai'],
                           columns=['pred_human', 'pred_ai']).to_string())
        tn, fp, fn, tp = cm.ravel()
        print(f'   false positives (human called ai) = {fp},  '
              f'false negatives (ai called human) = {fn}\n')

bert_table = result_table(BERT_RESULT, 'BERT, best configuration per dataset\n')

BERT, best configuration per dataset

    dataset  learning_rate  batch_size  weight_decay  epochs_run  val_f1  test_accuracy  test_precision  test_recall  test_f1
D1 DAIGT V2        0.00003          32           0.1           5  0.9957         0.9916          0.9916       0.9916   0.9916
     D2 HC3        0.00002          16           0.1           3  0.9939         0.9916          0.9917       0.9916   0.9916


In [7]:
show_confusion(BERT_RESULT)

D1 DAIGT V2 test confusion (rows true, cols predicted, order human, ai)
            pred_human  pred_ai
true_human        3475       44
true_ai             15     3464
   false positives (human called ai) = 44,  false negatives (ai called human) = 15

D2 HC3 test confusion (rows true, cols predicted, order human, ai)
            pred_human  pred_ai
true_human        5304       73
true_ai             17     5338
   false positives (human called ai) = 73,  false negatives (ai called human) = 17



## 2.7 What this notebook produced

For each dataset, three artefacts keyed by the configuration string
`full_{tag}_BERT_lr{lr}_bs{bs}_wd{wd}_s42`:

* `experiments/paper_scale/results/{key}.json` — the metrics quoted in the report,
* `experiments/paper_scale/probs/{key}.npz` — validation and test class probabilities,
* `experiments/paper_scale/models/{tag}_BERT/` — the fine-tuned weights and tokenizer.

Notebook 04 reads the `.npz` probabilities. Nothing is recomputed there, so the
ensemble is built from exactly these predictions.

In [8]:
for tag, rec in BERT_RESULT.items():
    print(f'{tag}  key={rec["key"]}')
    print(f'    results  {(RESULTS_DIR / (rec["key"] + ".json")).exists()}'
          f'   probs {(PROBS_DIR / (rec["key"] + ".npz")).exists()}'
          f'   model {(MODELS_DIR / f"{tag}_BERT").exists()}')

D1  key=full_D1_BERT_lr3e-05_bs32_wd0.1_s42
    results  True   probs True   model True
D2  key=full_D2_BERT_lr2e-05_bs16_wd0.1_s42
    results  True   probs True   model True
